
# Chapter 15: Top-P & Top-K — Sampling Experiment





1. What is Top-K?
Suppose the model predicts the next token:
Token	Probability
AI	35%
machine	25%
data	15%
model	10%
system	5%
banana	2%
car	1%


If:
top_k = 3


the model only considers the 3 highest-probability tokens:
AI
machine
data

Everything else is removed from consideration.

2. What is Top-P?
Top-P works differently.
Instead of saying:
"Keep exactly 10 tokens."

you say:
"Keep enough tokens until their cumulative probability reaches P."

For example:
AI        35%
machine   25%
data      15%
model     10%
system     5%
----------------
Total     90%

If:
top_p = 0.90


the model keeps these tokens because their cumulative probability reaches 90%.
Simple definition
Top-P = Keep the smallest group of tokens whose cumulative probability reaches P.

This is also called nucleus sampling.

| Parameter | Controls | Example |
|---|---|---|
| Temperature | How sharp/spread probability distribution is | `0.2`, `0.8` |
| Top-K | Number of candidate tokens | `10`, `50` |
| Top-P | Cumulative probability | `0.8`, `0.95` |

| | Top-K | Top-P |
|---|---|---|
| Controls | Number of tokens | Probability mass |
| Example | `top_k=50` | `top_p=0.9` |
| Candidate count | Fixed | Dynamic |
| Adapts to probability distribution | ❌ | ✅ |
| Common in modern APIs | Less common | More common |
| Good default | Sometimes | **Yes** |

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()


os.environ["GROQ_API_KEY"] = "YOUR API"


In [21]:
# Setting up environment variable

def setup_environment():
  """ Loads environment variables and check for the required API key"""

  load_dotenv()
  if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")


In [3]:
%pip install -U langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.0
    Uninstalling langchain-1.4.0:
      Successfully uninstalled langchain-1.4.0


In [4]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 10.5 MB/s eta 0:00:00


In [5]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

In [6]:
model = "openai/gpt-oss-20b"

# 1. First experiment — Top-P

In [22]:
prompt = "Give me a creative name for an AI research assistant."

top_p_values = [0.1, 0.5, 0.9, 1.0]

for p in top_p_values:

    model = llm.bind(top_p=p)

    response = model.invoke(prompt)

    print("TOP-P:", p)
    print("RESPONSE:")
    print(response.content)

TOP-P: 0.1
RESPONSE:
Here are a handful of creative names for an AI research assistant—each with a quick tagline to hint at its vibe:

| # | Name | Tagline / Why it fits |
|---|------|------------------------|
| 1 | **CogniCompanion** | “Your ever‑ready partner for deep dives into data.” |
| 2 | **InsightQuill** | “Turning raw research into polished prose.” |
| 3 | **SageSynth** | “Synthesizing knowledge with the wisdom of a sage.” |
| 4 | **QueryQuokka** | “Friendly, quick, and always ready to fetch answers.” |
| 5 | **NeuroNexus** | “The hub where neural networks meet human curiosity.” |
| 6 | **IdeaIris** | “Seeing the spectrum of possibilities in every dataset.” |
| 7 | **ScholarSynth** | “Your synthetic scholar for literature reviews and beyond.” |

Feel free to mix and match elements or tweak the names to better match your project’s tone!
TOP-P: 0.5
RESPONSE:
Here are a few creative, memorable names you could consider for an AI research assistant—each with a quick tagline to help

In [ ]:
# Running top values 5 time for each top p value

In [23]:
prompt = "Give me a creative name for an AI research assistant."

top_p_values = [0.1, 0.5, 0.9, 1.0]

for p in top_p_values:

    model = llm.bind(top_p=p)

    print("TOP-P:", p)

    for run in range(5):

        response = model.invoke(prompt)

        print(f"Run {run + 1}:")
        print(response.content)

TOP-P: 0.1
Run 1:
Here are a handful of creative names for an AI research assistant—each with a quick tagline to hint at its vibe:

| # | Name | Tagline / Why it fits |
|---|------|-----------------------|
| 1 | **ScholarSynth** | “Where curiosity meets computation.” |
| 2 | **InsightBot** | “Turning data into discovery.” |
| 3 | **QuantumQuill** | “Drafting the next frontier of knowledge.” |
| 4 | **EchoScholar** | “Amplifying every research whisper.” |
| 5 | **MuseMatrix** | “Your creative companion in the lab of ideas.” |
| 6 | **AetherMind** | “Navigating the ether of information.” |
| 7 | **NovaNavigator** | “Charting new stars in the research cosmos.” |

Feel free to mix and match elements or tweak the taglines to better match your project’s personality!
Run 2:
Here are a handful of creative names for an AI research assistant—each with a quick tagline to hint at its vibe:

| # | Name | Tagline / Why it fits |
|---|------|------------------------|
| 1 | **CogniCompanion** | “Your 

# 7. Compare Temperature + Top-P

In [27]:
# Low Temperature + Low Top-P = More Predicatble

In [28]:
prompt = "Give me a creative name for an AI research assistant."

experiments = [
    {"temperature": 0.2, "top_p": 0.5},
    {"temperature": 0.2, "top_p": 0.9},
    {"temperature": 0.8, "top_p": 0.5},
    {"temperature": 0.8, "top_p": 0.9},
]

for i in experiments:

    model = llm.bind(
        temperature=i["temperature"],
        top_p=i["top_p"]
    )

    response = model.invoke(prompt)

    print("TEMPERATURE:", i["temperature"])
    print("TOP-P:", i["top_p"])
    print("RESPONSE:")
    print(response.content)

TEMPERATURE: 0.2
TOP-P: 0.5
RESPONSE:
Here are a handful of creative names for an AI research assistant—each with a quick tagline to hint at its vibe:

| # | Name | Tagline / Quick Pitch |
|---|------|-----------------------|
| 1 | **CogniCompanion** | “Your ever‑learning partner in the maze of knowledge.” |
| 2 | **InsightBot** | “Turning data into clarity, one query at a time.” |
| 3 | **ResearchRaven** | “Swoops in with fresh angles and sharp insights.” |
| 4 | **ScholarSynth** | “Synthesizing scholarship so you can focus on discovery.” |
| 5 | **AetherMind** | “A cloud‑bound mind that never stops exploring.” |
| 6 | **NovaNavigator** | “Charting new frontiers in research, one star at a time.” |
| 7 | **QuantumQuill** | “Writing the next chapter of science with quantum precision.” |
| 8 | **EchoScholar** | “Amplifying your ideas, echoing them across disciplines.” |
| 9 | **SageSynth** | “Wisdom meets machine learning in a single, silent assistant.” |
|10 | **IdeaInfuser** | “Infusin

In [30]:
prompt = "Give me a creative name for an AI research assistant."

top_k_values = [1, 5, 20, 50]

for k in top_k_values:

    model = llm.bind(top_k=k)

    response = model.invoke(prompt)

    print("TOP-K:", k)
    print("RESPONSE:")
    print(response.content)

TypeError: Completions.create() got an unexpected keyword argument 'top_k'. Did you mean 'top_p'?

# My model do not support TOP K